# Import Library

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine

In [2]:
df = pd.read_excel("https://raw.githubusercontent.com/nikensaras/Data-Analyst-Project/main/FCMG/dataset%20FMCG.xlsx")

In [3]:
df.head(20)

,Invoice_No,Retailer_ID,Sales_Date,Region,City,SKU_Category,Brand,Units_Sold,Unit_Price,Discount_Pct,Sales_Value,COGS,Outlet_Type,Sales_Channel,Customer_Age,Loyalty_Flag,Return_Flag,Target_Sales
0,FMC-000001,RTL1965,21/08/2025,Jogja,Bandung,Food,brand b,1,797.24,0.50,914.85,912.85,General Trade,Distributor,67,N,Y,108312
1,FMC-000002,NaN,Aug-08-2025,Jakrta,Jakarta,Household,BrandA,3573,131.59,0.05,446662.52,242658.50,GT,Online,36,N,NaN,490592
2,FMC-000003,NaN,Sep-12-2024,Bandung,Yogyakarta,FOOD,Brand B,13,517.66,0.20,5383.66,3564.95,GT,Online,56,N,Y,432032
3,FMC-000004,RTL1193,Mar-31-2025,Bandng,Yogyakarta,Household,brand b,8,321.69,0.95,128.68,164.39,GT,Offline,67,N,N,71174
4,FMC-000005,RTL3648,Mar-26-2024,JKT,Bandung,Personal Care,BrandA,10,877.23,0.20,7017.84,5105.24,GT,Online,58,Y,Y,250452
5,FMC-000006,RTL3634,Nov-04-2024,Surabaya,Yogyakarta,FOOD,Brand-C,1,291.76,NaN,291.76,356.82,MT,Distributor,32,NaN,N,328783
6,FMC-000007,RTL3538,28/09/2024,Jakrta,Surabaya,Personal Care,Brand B,2,850.82,0.05,1616.56,1810.63,MT,Distributor,42,Y,Y,235390
7,FMC-000008,RTL2525,25/09/2023,JAKARTA,Yogyakarta,FOOD,BrandA,7,631.50,0.95,221.03,271.22,GT,Online,42,NaN,Y,101876
8,FMC-000009,RTL2805,2025-09-12,JKT,Bandng,Personal Care,BrandA,15,32.71,0.20,392.52,255.10,GT,Online,54,N,N,303087
9,FMC-000010,RTL4823,25/02/2024,Jakrta,Bandng,food,BrandA,15,74.13,0.75,277.99,213.14,Modern Trade,Online,37,Y,NaN,355643


# Connect SQL

In [ ]:
pip install sqlalchemy psycopg2-binary

In [ ]:
username = ""
password = ("")
host = ""
port = ""
database = ""

DATABASE_URL = (
    f"postgresql://{username}:{password}@{host}:{port}/{database}"
)

engine = create_engine(DATABASE_URL)


In [ ]:
df.to_sql('fmcg', con=engine, schema='public', if_exists='replace', index=False)

1000

# Info Data

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Invoice_No        10000 non-null  object        
 1   Retailer_ID       9356 non-null   object        
 2   Sales_Date        10000 non-null  datetime64[ns]
 3   City              10000 non-null  object        
 4   SKU_Category      10000 non-null  object        
 5   Brand             10000 non-null  object        
 6   Units_Sold        10000 non-null  int64         
 7   Unit_Price        10000 non-null  float64       
 8   Discount_Pct      10000 non-null  float64       
 9   Sales_Value       10000 non-null  float64       
 10  COGS              10000 non-null  float64       
 11  Outlet_Type       10000 non-null  object        
 12  Sales_Channel     10000 non-null  object        
 13  Customer_Age      10000 non-null  int64         
 14  Loyalty_Flag      6678 

# Cleaning Data

## Data Missing Values

In [ ]:
df.isnull().sum()

,0
Invoice_No,0
Retailer_ID,644
Sales_Date,0
Region,0
City,0
SKU_Category,0
Brand,0
Units_Sold,0
Unit_Price,0
Discount_Pct,822


## Isi Missing Values

In [ ]:
df.loc[df['Discount_Pct'].isna(), 'Discount_Pct'] = (
    1 - (
        df['Sales_Value'] /
        (df['Units_Sold'] * df['Unit_Price'])
    )
)

## Drop Kolom

In [ ]:
df.drop(columns='Region',inplace=True)

## Ubah Tipe Data

### Tipe Data Tanggal

In [ ]:
df['Sales_Date'] = pd.to_datetime(df['Sales_Date'], format='mixed', dayfirst=True)

### Ubah Tahun 2026 menjadi 2025

In [ ]:
df. loc[
  df ['Sales_Date'].dt.year == 2026,
  'Sales_Date'
] = df. loc[
    df ['Sales_Date']. dt. year == 2026,
  'Sales_Date'
].apply(lambda x: x.replace(year=2025))

## Writing Consistency

### City

In [ ]:
df['City'] = df['City'].replace({'Bandng': 'Bandung', 'JKT': 'Jakarta', 'Jkarta': 'Jakarta', 'jakrta': 'Jakarta', 'Jakrta': 'Jakarta', 'Bogor ':'Bogor', 'Jogja':'Yogyakarta', 'Yogyakrta':'Yogyakarta', 'Sby': 'Surabaya'})

### SKU Category

In [ ]:
df['SKU_Category'] = df['SKU_Category'].replace({'FOOD': 'Food', 'food': 'Food'})

### Brand

In [ ]:
df['Brand'] = df['Brand'].replace({'brand b': 'Brand B', 'BrandA': 'Brand A', 'Brand-C': 'Brand C'})

### Outlet Type

In [ ]:
df['Outlet_Type'] = df['Outlet_Type'].replace({'GT': 'General Trade', 'MT': 'Modern Trade'})

## Date Format

# Feature Engineer

## Gross Margin

In [ ]:
df['Gross_Margin'] = (df['Sales_Value'] - df['COGS']).round(2)

In [ ]:
df[df['Retailer_ID'].duplicated(keep=False)].sort_values('Retailer_ID')

,Invoice_No,Retailer_ID,Sales_Date,City,SKU_Category,Brand,Units_Sold,Unit_Price,Discount_Pct,Sales_Value,COGS,Outlet_Type,Sales_Channel,Customer_Age,Loyalty_Flag,Return_Flag,Target_Sales,Gross_Margin
5944,FMC-003410,RTL1000,Oct-21-2025,Jogja,Household,BrandA,20,810.11,0.20,12961.76,16841.39,General Trade,Online,60,N,NaN,78411,-3879.63
8862,FMC-008863,RTL1000,2024-09-30,Jogja,Food,Brand B,11,616.07,0.05,6437.93,4969.32,General Trade,Online,20,N,NaN,265983,1468.61
2282,FMC-002283,RTL1000,03/06/2024,Sby,Personal Care,BrandA,9,595.60,0.95,268.02,249.98,GT,Online,70,NaN,Y,245103,18.04
3803,FMC-003804,RTL1000,May-05-2024,Yogyakarta,Food,BrandA,18,450.99,0.95,405.89,261.27,MT,Online,67,Y,NaN,127075,144.62
1527,FMC-001528,RTL1000,Dec-06-2024,Jakrta,Food,Brand-C,11,999.40,0.15,9344.39,11294.40,General Trade,Offline,64,Y,NaN,462685,-1950.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9898,FMC-009899,NaN,2024-06-02,Jogja,food,Brand B,15,221.37,0.95,166.03,94.72,General Trade,Offline,24,N,NaN,173770,71.31
9901,FMC-009902,NaN,2026-03-23,Yogyakarta,Personal Care,brand b,4,325.31,0.20,1040.99,840.37,Modern Trade,Distributor,65,NaN,NaN,222812,200.62
9959,FMC-009960,NaN,28/01/2024,Jogja,Food,Brand B,6,309.00,0.75,463.50,420.49,MT,Offline,18,Y,NaN,43828,43.01
9963,FMC-009964,NaN,2026-01-16,Jakarta,Household,Brand B,11,857.16,0.95,471.44,351.33,GT,Offline,20,NaN,N,334090,120.11


## Achievement Rate

In [ ]:
df['Achievement_Rate'] = (
    df['Sales_Value']
    /
    df['Target_Sales']
) * 100

In [ ]:
df

,Invoice_No,Retailer_ID,Sales_Date,City,SKU_Category,Brand,Units_Sold,Unit_Price,Discount_Pct,Sales_Value,COGS,Outlet_Type,Sales_Channel,Customer_Age,Loyalty_Flag,Return_Flag,Target_Sales,Gross_Margin,Achievement_Rate
0,FMC-000001,RTL1965,21/08/2025,Bandung,Food,brand b,1,797.24,0.500000,914.85,912.85,General Trade,Distributor,67,N,Y,108312,2.00,0.844643
1,FMC-000002,NaN,Aug-08-2025,Jakarta,Household,BrandA,3573,131.59,0.050000,446662.52,242658.50,GT,Online,36,N,NaN,490592,204004.02,91.045618
2,FMC-000003,NaN,Sep-12-2024,Yogyakarta,FOOD,Brand B,13,517.66,0.200000,5383.66,3564.95,GT,Online,56,N,Y,432032,1818.71,1.246125
3,FMC-000004,RTL1193,Mar-31-2025,Yogyakarta,Household,brand b,8,321.69,0.950000,128.68,164.39,GT,Offline,67,N,N,71174,-35.71,0.180796
4,FMC-000005,RTL3648,Mar-26-2024,Bandung,Personal Care,BrandA,10,877.23,0.200000,7017.84,5105.24,GT,Online,58,Y,Y,250452,1912.60,2.802070
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,FMC-009996,RTL2985,Nov-24-2023,Bandung,Personal Care,Brand-C,6,68.15,0.950000,13.04,16.19,GT,Distributor,53,Y,Y,62889,-3.15,0.020735
9996,FMC-009997,RTL3712,Jul-19-2025,Bandng,Personal Care,Brand-C,3,709.54,0.100000,1915.76,1911.16,Modern Trade,Distributor,36,N,Y,477771,4.60,0.400979
9997,FMC-009998,RTL3880,28/02/2025,Jogja,FOOD,Brand-C,20,209.73,0.150000,3565.41,2447.72,General Trade,Distributor,65,Y,Y,417065,1117.69,0.854881
9998,FMC-009999,RTL3946,2025-06-12,Jakarta,Personal Care,Brand B,15,647.45,-1.248838,21840.15,15226.80,Modern Trade,Distributor,42,NaN,N,468822,6613.35,4.658516


## Age Group

In [ ]:
df['Age_Group'] = pd.cut(
    df['Customer_Age'],
    bins=[0,25,35,45,55,100],
    labels=[
        '18-25',
        '26-35',
        '36-45',
        '46-55',
        '55+'
    ]
)

In [ ]:
df

,Invoice_No,Retailer_ID,Sales_Date,City,SKU_Category,Brand,Units_Sold,Unit_Price,Discount_Pct,Sales_Value,COGS,Outlet_Type,Sales_Channel,Customer_Age,Loyalty_Flag,Return_Flag,Target_Sales,Gross_Margin,Achievement_Rate,Age_Group
0,FMC-000001,RTL1965,2025-08-21,Bandung,Food,brand b,1,797.24,0.500000,914.85,912.85,General Trade,Distributor,67,N,Y,108312,2.00,0.844643,55+
1,FMC-000002,NaN,2025-08-08,Jakarta,Household,BrandA,3573,131.59,0.050000,446662.52,242658.50,GT,Online,36,N,NaN,490592,204004.02,91.045618,36-45
2,FMC-000003,NaN,2024-09-12,Yogyakarta,FOOD,Brand B,13,517.66,0.200000,5383.66,3564.95,GT,Online,56,N,Y,432032,1818.71,1.246125,55+
3,FMC-000004,RTL1193,2025-03-31,Yogyakarta,Household,brand b,8,321.69,0.950000,128.68,164.39,GT,Offline,67,N,N,71174,-35.71,0.180796,55+
4,FMC-000005,RTL3648,2024-03-26,Bandung,Personal Care,BrandA,10,877.23,0.200000,7017.84,5105.24,GT,Online,58,Y,Y,250452,1912.60,2.802070,55+
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,FMC-009996,RTL2985,2023-11-24,Bandung,Personal Care,Brand-C,6,68.15,0.950000,13.04,16.19,GT,Distributor,53,Y,Y,62889,-3.15,0.020735,46-55
9996,FMC-009997,RTL3712,2025-07-19,Bandng,Personal Care,Brand-C,3,709.54,0.100000,1915.76,1911.16,Modern Trade,Distributor,36,N,Y,477771,4.60,0.400979,36-45
9997,FMC-009998,RTL3880,2025-02-28,Jogja,FOOD,Brand-C,20,209.73,0.150000,3565.41,2447.72,General Trade,Distributor,65,Y,Y,417065,1117.69,0.854881,55+
9998,FMC-009999,RTL3946,2025-06-12,Jakarta,Personal Care,Brand B,15,647.45,-1.248838,21840.15,15226.80,Modern Trade,Distributor,42,NaN,N,468822,6613.35,4.658516,36-45


Data start-end year

In [ ]:
df['Sales_Date'] = pd.to_datetime(df['Sales_Date'], errors='coerce')
min_date = df['Sales_Date'].min()
max_date = df['Sales_Date'].max()

print(f"Sales data starts on: {min_date.strftime('%B %Y')}")
print(f"Sales data ends on: {max_date.strftime('%B %Y')}")

Sales data starts on: September 2023
Sales data ends on: December 2025


# Insight

## Sales Channel berdasar Total Sales dan Profit

In [5]:
df = pd.read_csv("/content/insight 3.csv")
df.head()

,Sales_Channel,total_sales,total_profit
0,Online,7.766275e+08,1.402098e+08
1,Distributor,9.927261e+07,8.568497e+06
2,Offline,8.531993e+07,4.138003e+06


## RFM

In [7]:
df = pd.read_csv("/content/rfm.csv")
df.head()

,Retailer_ID,recency,frequency,monetary,r_score,f_score,m_score,total_rfm_score,segmentasi
0,RTL3430,27,4,13001.54,4,1,1,6,hibernating
1,RTL3556,278,4,12419.30,2,1,1,4,lost
2,RTL2682,365,1,257.36,1,1,1,3,lost
3,RTL4825,658,1,4855.38,1,1,1,3,lost
4,RTL2236,222,4,3979.26,2,1,1,4,lost


## Segementasi Customer

In [6]:
df = pd.read_csv("/content/inssight 7.csv")
df.head()

,segmentasi,total_customers
0,lost,2328
1,hibernating,1131
2,at risk,134
3,loyal,28


## Normal vs Peak Season

In [8]:
df = pd.read_csv("/content/insight 4.csv")
df.head()

,month,total_sales,season_type
0,Sep 2023,18579445.46,Low Period
1,Oct 2023,4995659.47,Low Period
2,Nov 2023,14492093.30,Low Period
3,Dec 2023,4398141.07,Low Period
4,Jan 2024,6510526.23,Low Period
